# 03 — Baseline Model Comparison

Compares Logistic Regression, HistGradientBoosting, and Isolation Forest on the fraud detection task with proper time-based splitting.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

from src.feature_store import build_features, MODEL_FEATURES
from src.models.graph_features import build_graph_features
from src.rules_engine import apply_rules
from src.models.logistic import build_logistic
from src.models.xgb_model import build_hgb
from src.models.anomaly import build_isolation_forest
from src.config import load_config

cfg = load_config("../configs/default.yaml")

## Prepare data

In [ ]:
train = pd.read_csv("../data/synthetic/train.csv")
test = pd.read_csv("../data/synthetic/test.csv")
accounts = pd.read_csv("../data/synthetic/accounts.csv")
customers = pd.read_csv("../data/synthetic/customers.csv")

train_f = build_graph_features(build_features(train, accounts, customers))
test_f = build_graph_features(build_features(test, accounts, customers))

X_train = train_f[MODEL_FEATURES]
y_train = train_f["label_fraud"].astype(int)
X_test = test_f[MODEL_FEATURES]
y_test = test_f["label_fraud"].astype(int)

print(f"Train: {X_train.shape}  fraud rate: {y_train.mean():.4f}")
print(f"Test:  {X_test.shape}  fraud rate: {y_test.mean():.4f}")

## Train models

In [ ]:
logit = build_logistic(**cfg["models"]["logistic"])
hgb = build_hgb(**cfg["models"]["hgb"])
iforest = build_isolation_forest(**cfg["models"]["isolation_forest"])

logit.fit(X_train, y_train)
hgb.fit(X_train, y_train)
iforest.fit(X_train)

logit_scores = logit.predict_proba(X_test)[:, 1]
hgb_scores = hgb.predict_proba(X_test)[:, 1]
iforest_scores = -iforest.score_samples(X_test)
iforest_norm = (iforest_scores - iforest_scores.min()) / (iforest_scores.max() - iforest_scores.min() + 1e-12)

## ROC and PR curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, scores, color in [("Logistic", logit_scores, "steelblue"),
                              ("HGB", hgb_scores, "salmon"),
                              ("IForest", iforest_norm, "green")]:
    auc = roc_auc_score(y_test, scores)
    fpr, tpr, _ = roc_curve(y_test, scores)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})", color=color)

    prauc = average_precision_score(y_test, scores)
    prec, rec, _ = precision_recall_curve(y_test, scores)
    axes[1].plot(rec, prec, label=f"{name} (PR-AUC={prauc:.4f})", color=color)

axes[0].plot([0,1],[0,1], "k--", alpha=0.3)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC Curve"); axes[0].legend()
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("PR Curve"); axes[1].legend()
plt.tight_layout()
plt.show()

## Summary metrics

In [ ]:
results = []
for name, scores in [("Logistic", logit_scores), ("HGB", hgb_scores), ("IForest", iforest_norm)]:
    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, scores),
        "pr_auc": average_precision_score(y_test, scores),
    })
pd.DataFrame(results).set_index("model")

## Key findings

- HistGradientBoosting outperforms Logistic Regression on both ROC-AUC and PR-AUC.
- Isolation Forest captures anomalous patterns but is not calibrated as a probability — useful as an ensemble component.
- The models are trained on chronologically earlier data and tested on later data, avoiding temporal leakage.